# Object Detection Workshop

## Set-up

In [ ]:
# setup

In [ ]:
# ---- Load the COCO annotation file for a split -------------------------
#   DATA_ROOT/<split>/_annotations.coco.json   +   the image files
def load_split(split):
    with open(os.path.join(DATA_ROOT, split, "_annotations.coco.json")) as f:
        return json.load(f)

train_coco = load_split("train")

real_cats = sorted([c for c in train_coco["categories"] if c["supercategory"] != "none"],
                   key=lambda c: c["id"])
CAT_IDS         = [c["id"]   for c in real_cats]   # e.g. [1, 2, 3]
CLASS_NAMES     = [c["name"] for c in real_cats]   # e.g. ['green_ball','red_ball','yellow_ball']
NUM_CLASSES_CLS = len(CLASS_NAMES)                 # 3  -> for the CNN classifier (0-indexed labels)
NUM_CLASSES_DET = len(CLASS_NAMES) + 1             # 4  -> for detectors (+1 because id 0 = BACKGROUND)

catid_to_clsidx = {cid: i for i, cid in enumerate(CAT_IDS)}   # {1:0, 2:1, 3:2}

print("Classes:", CLASS_NAMES)
print("CNN classifier num_classes:", NUM_CLASSES_CLS)
print("Detector num_classes (include background):", NUM_CLASSES_DET)

In [ ]:
for split in ["train", "valid", "test"]:
    c = load_split(split)
    print(f"{split:6s}: {len(c['images']):3d} images, {len(c['annotations']):3d} boxes")

In [ ]:
def show_gt(split, n=3): # n = number of shown image (default=3)
    coco = load_split(split)
    by_img = {}
    for a in coco["annotations"]:
        by_img.setdefault(a["image_id"], []).append(a)
    id_to_name = {c["id"]: c["name"] for c in coco["categories"]}

    imgs = coco["images"][:n]
    fig, axes = plt.subplots(1, len(imgs), figsize=(5 * len(imgs), 5))
    if len(imgs) == 1: axes = [axes]
    for ax, info in zip(axes, imgs):
        img = Image.open(os.path.join(DATA_ROOT, split, info["file_name"])).convert("RGB")
        ax.imshow(img); ax.axis("off"); ax.set_title(info["file_name"][:16])
        for a in by_img.get(info["id"], []):
            x, y, w, h = a["bbox"]
            ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, color="lime", lw=2))
            ax.text(x, y - 4, id_to_name[a["category_id"]], color="lime", fontsize=9, weight="bold")
    plt.tight_layout(); plt.show()

show_gt("train", 3)

## Section 1 — Custom CNN: **classification**

In [ ]:
#custom CNN :classification

In [ ]:
# build CNN achitecture

In [ ]:
# --- Training loop -----------


In [ ]:
# --- Evaluate: accuracy + a confusion matrix ----------------------------
@torch.no_grad()
def cnn_accuracy(loader):
    cnn.eval()
    correct = total = 0
    cm = np.zeros((NUM_CLASSES_CLS, NUM_CLASSES_CLS), dtype=int)  # [true, pred]
    for imgs, labels in loader:
        preds = cnn(imgs.to(DEVICE)).argmax(1).cpu().numpy()
        labs = labels.numpy()
        for p, t in zip(preds, labs):
            cm[t, p] += 1
        correct += (preds == labs).sum(); total += len(labs)
    return correct / total, cm

acc, cm = cnn_accuracy(valid_loader_cls)
print(f"valid accuracy: {acc:.2%}")

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES_CLS)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(NUM_CLASSES_CLS)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Confusion Matrix")
for i in range(NUM_CLASSES_CLS):
    for j in range(NUM_CLASSES_CLS):
        ax.text(j, i, cm[i, j], ha="center", va="center")
plt.tight_layout(); plt.show()

In [ ]:
# --- prediction on test images --------------


## Shared detector helpers (support Faster R-CNN and SSD)

In [ ]:
class CocoDetectionDataset(Dataset):
    def __init__(self, split):
        self.split = split
        self.coco = load_split(split)
        self.images = self.coco["images"]
        self.by_img = {}
        for a in self.coco["annotations"]:
            if a["category_id"] in catid_to_clsidx:           
                self.by_img.setdefault(a["image_id"], []).append(a)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        info = self.images[idx]
        img = Image.open(os.path.join(DATA_ROOT, self.split, info["file_name"])).convert("RGB")
        boxes, labels = [], []
        for a in self.by_img.get(info["id"], []):
            x, y, w, h = a["bbox"]
            boxes.append([x, y, x + w, y + h])   
            labels.append(a["category_id"])      
        boxes  = torch.as_tensor(boxes,  dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        return TF.to_tensor(img), {"boxes": boxes, "labels": labels}

def collate_fn(batch):
    return tuple(zip(*batch))

train_det = CocoDetectionDataset("train")
valid_det = CocoDetectionDataset("valid")
test_det  = CocoDetectionDataset("test")

train_loader_det = DataLoader(train_det, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
test_loader_det  = DataLoader(test_det,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print("detector datasets:", len(train_det), len(valid_det), len(test_det))

In [ ]:
# These three helpers are reused UNCHANGED by both detectors below.
from torchmetrics.detection import MeanAveragePrecision

def train_one_epoch(model, loader, optimizer):
    model.train()
    total = 0.0
    for images, targets in loader:
        images  = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)       # forward -> {'loss_classifier': ..., ...}
        loss = sum(loss_dict.values())
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def evaluate_map(model, loader):
    """Run inference over a loader and compute COCO-style mAP with torchmetrics."""
    model.eval()
    metric = MeanAveragePrecision()          
    for images, targets in loader:
        images = [img.to(DEVICE) for img in images]
        preds = model(images)                # predictions
        preds = [{k: v.cpu() for k, v in p.items()} for p in preds]
        metric.update(preds, list(targets))
    return metric.compute()

@torch.no_grad()
def visualize_predictions(model, dataset, n=4, score_threshold=SCORE_THRESHOLD):
    """Draw predicted boxes on the REAL images. """
    model.eval()
    idxs = list(range(min(n, len(dataset))))
    fig, axes = plt.subplots(1, len(idxs), figsize=(6 * len(idxs), 6))
    if len(idxs) == 1: axes = [axes]
    for ax, i in zip(axes, idxs):
        img, _ = dataset[i]
        pred = model([img.to(DEVICE)])[0]
        ax.imshow(img.permute(1, 2, 0).numpy()); ax.axis("off")
        keep = pred["scores"] >= score_threshold
        for box, lab, sc in zip(pred["boxes"][keep].cpu(), pred["labels"][keep].cpu(), pred["scores"][keep].cpu()):
            x1, y1, x2, y2 = box.tolist()
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="red", lw=2))
            name = CLASS_NAMES[catid_to_clsidx[int(lab)]] if int(lab) in catid_to_clsidx else str(int(lab))
            ax.text(x1, y1 - 4, f"{name} {sc:.2f}", color="red", fontsize=10, weight="bold")
    plt.tight_layout(); plt.show()

@torch.no_grad()
def inference_time(model, dataset, n=4):
    model.eval()
    imgs = [dataset[i][0].to(DEVICE) for i in range(min(n, len(dataset)))]
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for im in imgs:
        model([im])
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    return (time.time() - t0) / len(imgs)

## Section 2 — Faster R-CNN: 

In [ ]:
# build Faster R-CNN model

In [ ]:
# Inference on the test set: draw predicted boxes, measure mAP

## Section 3 — SSD300: 

In [ ]:
# build SSD300 model

In [ ]:
# SSD helpher

In [ ]:
# --- model performance ---------
NUM_EPOCHS = 2
frcnn_t = inference_time(frcnn, test_det, n=4)
ssd_t   = inference_time(ssd,   test_det, n=4)

print(f"{'model':16s}{'mAP':>8s}{'mAP@50':>10s}{'sec/img':>10s}")
print("-" * 44)
print(f"{'Faster R-CNN':16s}{float(frcnn_map['map']):8.3f}{float(frcnn_map['map_50']):10.3f}{frcnn_t:10.4f}")
print(f"{'SSD300':16s}{float(ssd_map['map']):8.3f}{float(ssd_map['map_50']):10.3f}{ssd_t:10.4f}")
print()
print("Reminder: with only", NUM_EPOCHS, "fine-tune epochs and 4 test images these")
print("numbers are NOISY. Look at the TREND (speed vs accuracy), not the exact values.")

**Homework ideas**
1. Train the detectors for more epochs and watch mAP improve.
2. Change `SCORE_THRESHOLD` and see how the drawn boxes change (precision vs recall).
3. Add data augmentation (random flips) to the detection `Dataset`.
4. Try another torchvision model detector (e.g. `retinanet_resnet50_fpn`) —